# Thesis Evaluation — Multi-Model Agentic Fashion System

This notebook produces the quantitative results for comparing **three orchestration modes**:

| Mode | Orchestrator | Synthesizer |
|------|-------------|-------------|
| A    | Gemini 2.0 Flash | Gemini 2.0 Flash |
| B    | Gemini 2.0 Flash (orchestrator) | GPT-4o (synthesizer) |
| C    | GPT-4o (orchestrator) | Claude 3.5 Sonnet (synthesizer) |

**Evaluation axes:**
1. Token cost & efficiency
2. Behavioural accuracy (SR, SCR, QRR, GAS)
3. Cost-Efficiency Score (CES)
4. Statistical significance tests

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from psycopg2.extras import RealDictCursor
from scipy import stats

# Style
sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

# Database connection
conn = psycopg2.connect(
    host=os.getenv('PGHOST', 'localhost'),
    port=int(os.getenv('PGPORT', '5432')),
    dbname=os.getenv('PGDATABASE', 'fashion_rag'),
    user=os.getenv('PGUSER', 'fashion_user'),
    password=os.getenv('PGPASSWORD', ''),
    connect_timeout=5,
)
print('Connected to PostgreSQL')

## 1 · Token Cost Analysis

In [ ]:
# ── Pricing constants (USD per 1M tokens) ──────────────────────────
PRICING = {
    'gemini': {'input': 0.075, 'output': 0.30},
    'gpt':    {'input': 2.50,  'output': 10.00},
    'claude': {'input': 3.00,  'output': 15.00},
}

# Pull mode_cost_summary view
df_cost = pd.read_sql('SELECT * FROM mode_cost_summary ORDER BY orchestration_mode;', conn)
display(df_cost)

In [ ]:
# ── Per-turn cost bar chart ─────────────────────────────────────────
if not df_cost.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Average tokens per turn
    ax1 = axes[0]
    ax1.bar(df_cost['orchestration_mode'], df_cost['avg_total_tokens'], color=sns.color_palette('pastel'))
    ax1.set_title('Avg Total Tokens per Turn')
    ax1.set_xlabel('Orchestration Mode')
    ax1.set_ylabel('Tokens')

    # Average USD per turn
    ax2 = axes[1]
    ax2.bar(df_cost['orchestration_mode'], df_cost['avg_usd_per_turn'].astype(float), color=sns.color_palette('muted'))
    ax2.set_title('Avg USD Cost per Turn')
    ax2.set_xlabel('Orchestration Mode')
    ax2.set_ylabel('USD')
    ax2.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

    plt.tight_layout()
    plt.savefig('analysis/token_cost_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print('No cost data available yet — run sessions first.')

## 2 · Behavioural Accuracy Metrics

In [ ]:
# ── Pull per-session behaviour data ─────────────────────────────────
q_behaviour = """
WITH session_modes AS (
    SELECT
        s.session_id,
        s.preferred_model,
        COALESCE(st.orchestration_mode, 'unknown') AS orchestration_mode,
        s.gender,
        s.gender_hint_enabled
    FROM user_sessions s
    LEFT JOIN session_token_summary st USING (session_id)
)
SELECT
    sm.orchestration_mode,
    sm.session_id,
    sm.gender,
    sm.gender_hint_enabled,
    (SELECT COUNT(*) FROM product_impressions pi WHERE pi.session_id = sm.session_id) AS impressions,
    (SELECT COUNT(*) FROM product_clicks pc WHERE pc.session_id = sm.session_id) AS clicks,
    (SELECT COUNT(*) FROM selected_items si WHERE si.session_id = sm.session_id) AS selections,
    (SELECT COUNT(*) FROM product_intents pit WHERE pit.session_id = sm.session_id AND pit.intent_type = 'will_buy') AS will_buy,
    (SELECT COUNT(*) FROM product_intents pit2 WHERE pit2.session_id = sm.session_id AND pit2.intent_type = 'not_for_me') AS not_for_me,
    (SELECT COUNT(*) FROM user_orders uo WHERE uo.session_id = sm.session_id) AS orders,
    (SELECT COUNT(DISTINCT search_query) FROM product_impressions pi3
        WHERE pi3.session_id = sm.session_id AND pi3.search_query <> '') AS distinct_queries
FROM session_modes sm;
"""
df_sessions = pd.read_sql(q_behaviour, conn)
print(f'Total sessions: {len(df_sessions)}')
display(df_sessions.head())

In [ ]:
# ── Compute per-session derived metrics ─────────────────────────────
df = df_sessions.copy()
df['sr']  = df['selections'] / df['impressions'].replace(0, np.nan)
df['scr'] = df['will_buy']   / df['selections'].replace(0, np.nan)
df['qrr'] = (df['distinct_queries'] >= 2).astype(int)
df['converted'] = (df['orders'] > 0).astype(int)

# ── Aggregate by orchestration mode ─────────────────────────────────
agg = df.groupby('orchestration_mode').agg(
    n_sessions=('session_id', 'count'),
    mean_sr=('sr', 'mean'),
    mean_scr=('scr', 'mean'),
    mean_qrr=('qrr', 'mean'),
    conversion_rate=('converted', 'mean'),
).round(4)
display(agg)

In [ ]:
# ── Radar chart of accuracy metrics ─────────────────────────────────
if not agg.empty and len(agg) > 1:
    categories = ['SR', 'SCR', 'QRR', 'Conversion']
    modes = agg.index.tolist()

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]

    colors = sns.color_palette('Set2', len(modes))
    for i, mode in enumerate(modes):
        values = [
            agg.loc[mode, 'mean_sr'],
            agg.loc[mode, 'mean_scr'],
            agg.loc[mode, 'mean_qrr'],
            agg.loc[mode, 'conversion_rate'],
        ]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=mode, color=colors[i])
        ax.fill(angles, values, alpha=0.15, color=colors[i])

    ax.set_thetagrids([a * 180 / np.pi for a in angles[:-1]], categories)
    ax.set_title('Behavioural Accuracy by Orchestration Mode', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig('analysis/accuracy_radar.png', bbox_inches='tight')
    plt.show()
else:
    print('Need >= 2 modes with data for radar chart.')

## 3 · Gender-Appropriate Selection (GAS)

In [ ]:
# ── Category gender inference ───────────────────────────────────────
MALE_CATS   = {'Longsleeve', 'T-Shirt', 'Shirt', 'Hoodie', 'Shorts', 'Pants', 'Blazer', 'Polo'}
FEMALE_CATS = {'Dress', 'Skirt', 'Blouse', 'Top'}

q_gas = """
WITH session_modes AS (
    SELECT s.session_id, s.gender, s.gender_hint_enabled,
           COALESCE(st.orchestration_mode, 'unknown') AS orchestration_mode
    FROM user_sessions s
    LEFT JOIN session_token_summary st USING (session_id)
    WHERE s.gender IS NOT NULL
)
SELECT sm.session_id, sm.gender, sm.gender_hint_enabled,
       sm.orchestration_mode, si.label
FROM session_modes sm
JOIN selected_items si USING (session_id);
"""
df_gas_raw = pd.read_sql(q_gas, conn)

def classify_match(row):
    if row['gender'] == 'male' and row['label'] in MALE_CATS:
        return 'match'
    elif row['gender'] == 'male' and row['label'] in FEMALE_CATS:
        return 'mismatch'
    elif row['gender'] == 'female' and row['label'] in FEMALE_CATS:
        return 'match'
    elif row['gender'] == 'female' and row['label'] in MALE_CATS:
        return 'mismatch'
    return 'unisex'

if not df_gas_raw.empty:
    df_gas_raw['gender_match'] = df_gas_raw.apply(classify_match, axis=1)
    df_gendered = df_gas_raw[df_gas_raw['gender_match'] != 'unisex']

    gas_table = df_gendered.groupby(['gender_hint_enabled', 'orchestration_mode']).agg(
        gendered_selections=('gender_match', 'count'),
        matched=('gender_match', lambda x: (x == 'match').sum()),
    )
    gas_table['gas'] = (gas_table['matched'] / gas_table['gendered_selections']).round(4)
    display(gas_table)
else:
    print('No gender-annotated sessions yet.')

## 4 · Cost-Efficiency Score (CES)

In [ ]:
# CES = (α · Norm_SR + β · Norm_SCR + γ · Norm_CR) / Norm_Cost
# Default weights: α=0.4, β=0.3, γ=0.3
ALPHA, BETA, GAMMA = 0.4, 0.3, 0.3

if not agg.empty:
    ces = agg.copy()

    # Min-max normalise (0–1)
    for col in ['mean_sr', 'mean_scr', 'conversion_rate']:
        cmin, cmax = ces[col].min(), ces[col].max()
        if cmax > cmin:
            ces[f'norm_{col}'] = (ces[col] - cmin) / (cmax - cmin)
        else:
            ces[f'norm_{col}'] = 1.0

    # Merge cost data
    if not df_cost.empty:
        cost_map = df_cost.set_index('orchestration_mode')['avg_usd_per_turn'].astype(float)
        ces['avg_cost'] = ces.index.map(lambda m: cost_map.get(m, np.nan))
        cmin, cmax = ces['avg_cost'].min(), ces['avg_cost'].max()
        if cmax > cmin:
            ces['norm_cost'] = (ces['avg_cost'] - cmin) / (cmax - cmin)
        else:
            ces['norm_cost'] = 1.0

        ces['CES'] = (
            ALPHA * ces['norm_mean_sr']
            + BETA * ces['norm_mean_scr']
            + GAMMA * ces['norm_conversion_rate']
        ) / ces['norm_cost'].replace(0, 0.001)  # avoid div-by-zero

        display(ces[['n_sessions', 'mean_sr', 'mean_scr', 'conversion_rate', 'avg_cost', 'CES']].round(4))
    else:
        print('No cost data to compute CES.')
else:
    print('No accuracy data to compute CES.')

## 5 · Statistical Significance Tests

In [ ]:
# ── 5a. Chi-square test on conversion rates ────────────────────────
if not df.empty and df['orchestration_mode'].nunique() >= 2:
    contingency = pd.crosstab(df['orchestration_mode'], df['converted'])
    if contingency.shape[1] == 2:
        chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
        print(f'Chi-square test (conversion ~ mode):')
        print(f'  χ² = {chi2:.4f}, p = {p_chi:.6f}, dof = {dof}')
        if p_chi < 0.05:
            print('  → Statistically significant at α=0.05')
        else:
            print('  → NOT statistically significant at α=0.05')
    else:
        print('Insufficient variation for chi-square (all sessions converted or none).')
else:
    print('Need >= 2 modes for chi-square test.')

In [ ]:
# ── 5b. Kruskal-Wallis test on SR ──────────────────────────────────
if not df.empty and df['orchestration_mode'].nunique() >= 2:
    groups = [g['sr'].dropna().values for _, g in df.groupby('orchestration_mode')]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        h_stat, p_kw = stats.kruskal(*groups)
        print(f'Kruskal-Wallis test (SR ~ mode):')
        print(f'  H = {h_stat:.4f}, p = {p_kw:.6f}')
    else:
        print('Not enough non-empty groups for Kruskal-Wallis.')
else:
    print('Need >= 2 modes for KW test.')

In [ ]:
# ── 5c. Bootstrap 95% CI for mean SR per mode ──────────────────────
def bootstrap_ci(data, n_boot=10000, ci=0.95):
    """Return (lower, upper) bootstrap CI for the mean."""
    if len(data) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(42)
    means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.quantile(means, alpha), np.quantile(means, 1 - alpha)

if not df.empty:
    print('Bootstrap 95% CI for mean SR:')
    for mode, g in df.groupby('orchestration_mode'):
        vals = g['sr'].dropna().values
        lo, hi = bootstrap_ci(vals)
        print(f'  {mode}: [{lo:.4f}, {hi:.4f}]  (n={len(vals)})')

In [ ]:
# ── 5d. Cliff's delta (effect size) for Mode A vs others ───────────
def cliffs_delta(x, y):
    """Compute Cliff's delta between x and y."""
    x, y = np.asarray(x), np.asarray(y)
    n = len(x) * len(y)
    if n == 0:
        return np.nan
    more = np.sum(x[:, None] > y[None, :])
    less = np.sum(x[:, None] < y[None, :])
    return (more - less) / n

modes = df['orchestration_mode'].unique()
if len(modes) >= 2:
    baseline = modes[0]
    base_vals = df.loc[df['orchestration_mode'] == baseline, 'sr'].dropna().values
    print(f"Cliff's delta (SR) — baseline: {baseline}")
    for m in modes[1:]:
        other_vals = df.loc[df['orchestration_mode'] == m, 'sr'].dropna().values
        d = cliffs_delta(base_vals, other_vals)
        magnitude = 'negligible'
        if abs(d) >= 0.474:
            magnitude = 'large'
        elif abs(d) >= 0.33:
            magnitude = 'medium'
        elif abs(d) >= 0.147:
            magnitude = 'small'
        print(f'  vs {m}: δ = {d:.4f} ({magnitude})')
else:
    print('Need >= 2 modes.')

## 6 · Summary Table for Thesis

In [ ]:
# ── Combined results summary ────────────────────────────────────────
if not agg.empty:
    summary = agg[['n_sessions', 'mean_sr', 'mean_scr', 'mean_qrr', 'conversion_rate']].copy()
    summary.columns = ['Sessions', 'SR', 'SCR', 'QRR', 'CR']
    
    if not df_cost.empty:
        cost_map = df_cost.set_index('orchestration_mode')[['avg_total_tokens', 'avg_usd_per_turn']]
        cost_map.columns = ['Avg Tokens/Turn', 'Avg USD/Turn']
        cost_map['Avg USD/Turn'] = cost_map['Avg USD/Turn'].astype(float)
        summary = summary.join(cost_map, how='left')
    
    if 'CES' in ces.columns:
        summary['CES'] = ces['CES'].round(4)
    
    print('\n════════════════ THESIS SUMMARY TABLE ════════════════')
    display(summary.round(4))
    
    # Export to CSV for LaTeX inclusion
    summary.to_csv('analysis/thesis_summary_table.csv')
    print('\nExported to analysis/thesis_summary_table.csv')
else:
    print('No data — run evaluation sessions first.')

## 7 · Direct vs ReAct Pipeline Comparison

Compares the **Direct** pipeline (Intent Classification → deterministic search) against the  
**ReAct** pipeline (tool-calling loop via Gemini) on 40 annotated retrieval queries.

Metrics computed per query: Hit@1, Hit@3, Hit@6, MRR, NDCG@6, latency (ms), LLM call count.

In [ ]:
# ── 7a. Load eval_results from Postgres ─────────────────────────────────────
q_eval = """
SELECT
    er.orchestration_mode,
    er.hit_at_1,  er.hit_at_3,  er.hit_at_6,
    er.reciprocal_rank, er.ndcg_at_6,
    er.latency_ms, er.llm_call_count, er.total_tokens,
    eq.difficulty, eq.category, eq.language
FROM eval_results er
JOIN eval_queries eq ON eq.id = er.eval_query_id
ORDER BY er.eval_query_id, er.orchestration_mode, er.run_at;
"""
df_eval = pd.read_sql(q_eval, conn)
print(f"eval_results rows: {len(df_eval)}")
print(df_eval['orchestration_mode'].value_counts().to_string())
df_eval.head()

In [ ]:
# ── 7b. Aggregate metrics per pipeline mode ─────────────────────────────────
if not df_eval.empty:
    eval_agg = df_eval.groupby('orchestration_mode').agg(
        n_queries=('hit_at_6', 'count'),
        hit_at_1=('hit_at_1', 'mean'),
        hit_at_3=('hit_at_3', 'mean'),
        hit_at_6=('hit_at_6', 'mean'),
        mrr=('reciprocal_rank', 'mean'),
        ndcg_at_6=('ndcg_at_6', 'mean'),
        avg_latency_ms=('latency_ms', 'mean'),
        avg_llm_calls=('llm_call_count', 'mean'),
        avg_total_tokens=('total_tokens', 'mean'),
    ).round(4)
    print("\n━━━ Pipeline Retrieval Metrics ━━━")
    display(eval_agg)
else:
    print("No eval_results data yet — run evaluation.run_comparison first.")
    eval_agg = pd.DataFrame()

In [ ]:
# ── 7c. Bar chart: Hit@K, MRR, NDCG@6 ──────────────────────────────────────
if not eval_agg.empty:
    metrics = ["hit_at_1", "hit_at_3", "hit_at_6", "mrr", "ndcg_at_6"]
    labels  = ["Hit@1", "Hit@3", "Hit@6", "MRR", "NDCG@6"]
    modes = eval_agg.index.tolist()
    x = np.arange(len(metrics))
    width = 0.35
    colors = sns.color_palette("Set2", len(modes))

    fig, ax = plt.subplots(figsize=(13, 5))
    for i, mode in enumerate(modes):
        vals = [eval_agg.loc[mode, m] for m in metrics]
        bars = ax.bar(x + i * width - width * (len(modes)-1)/2, vals,
                      width, label=mode.capitalize(), color=colors[i], alpha=0.88)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Direct vs ReAct — Retrieval Metrics (40 queries)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("analysis/pipeline_retrieval_comparison.png", bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# ── 7d. Latency & LLM call count ────────────────────────────────────────────
if not eval_agg.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    palette = sns.color_palette("Set2", len(eval_agg))

    ax1 = axes[0]
    ax1.bar(eval_agg.index, eval_agg["avg_latency_ms"], color=palette)
    ax1.set_title("Avg Latency (ms)")
    ax1.set_ylabel("ms")
    for i, (mode, val) in enumerate(eval_agg["avg_latency_ms"].items()):
        ax1.text(i, val + 2, f"{val:.0f}", ha="center", fontsize=10)

    ax2 = axes[1]
    ax2.bar(eval_agg.index, eval_agg["avg_llm_calls"], color=palette)
    ax2.set_title("Avg LLM Calls per Query")
    ax2.set_ylabel("calls")
    for i, (mode, val) in enumerate(eval_agg["avg_llm_calls"].items()):
        ax2.text(i, val + 0.02, f"{val:.2f}", ha="center", fontsize=10)

    ax3 = axes[2]
    ax3.bar(eval_agg.index, eval_agg["avg_total_tokens"], color=palette)
    ax3.set_title("Avg Total Tokens per Query")
    ax3.set_ylabel("tokens")
    for i, (mode, val) in enumerate(eval_agg["avg_total_tokens"].items()):
        ax3.text(i, val + 5, f"{val:.0f}", ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig("analysis/pipeline_cost_latency.png", bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# ── 7e. Breakdown by difficulty: NDCG@6 ─────────────────────────────────────
if not df_eval.empty:
    diff_pivot = df_eval.groupby(["orchestration_mode", "difficulty"])["ndcg_at_6"].mean().unstack("difficulty")
    diff_pivot = diff_pivot.reindex(columns=["easy", "medium", "hard"])

    diff_pivot.plot(kind="bar", figsize=(9, 5), colormap="Set2", edgecolor="white")
    plt.title("NDCG@6 by Difficulty — Direct vs ReAct")
    plt.ylabel("NDCG@6")
    plt.xlabel("Pipeline Mode")
    plt.xticks(rotation=0)
    plt.legend(title="Difficulty", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("analysis/pipeline_difficulty_ndcg.png", bbox_inches="tight", dpi=150)
    plt.show()
    display(diff_pivot.round(4))

## 8 · Statistical Tests — Direct vs ReAct

Tests use **paired** observations (same 40 queries evaluated under both modes).

| Test | Metric | Method |
|------|--------|--------|
| McNemar's | Hit@6 (binary) | Paired binary |
| Wilcoxon signed-rank | MRR, NDCG@6 | Paired continuous |
| Cliff's delta | MRR, NDCG@6 | Effect size |
| Bootstrap 95% CI | MRR, NDCG@6 | 10,000 resamples |

In [ ]:
# ── 8a. Pivot to paired format ───────────────────────────────────────────────
if not df_eval.empty and 'direct' in df_eval['orchestration_mode'].values \
        and 'react' in df_eval['orchestration_mode'].values:
    # Average over multiple runs per (query, mode)
    paired = df_eval.groupby(["eval_query_id", "orchestration_mode"])[
        ["hit_at_6", "reciprocal_rank", "ndcg_at_6"]
    ].mean().unstack("orchestration_mode")
    paired.columns = ["_".join(c) for c in paired.columns]
    paired = paired.dropna()
    print(f"Paired query count: {len(paired)}")
    display(paired.head())
else:
    print("Need both 'direct' and 'react' results in eval_results. Run run_comparison.py first.")
    paired = pd.DataFrame()

In [ ]:
# ── 8b. McNemar's test on Hit@6 ─────────────────────────────────────────────
from scipy.stats import binom as binom_dist

if not paired.empty:
    hit_direct = paired["hit_at_6_direct"].round().astype(int)
    hit_react  = paired["hit_at_6_react"].round().astype(int)

    b = ((hit_react == 1) & (hit_direct == 0)).sum()
    c = ((hit_react == 0) & (hit_direct == 1)).sum()
    n_pairs = len(paired)

    if b + c == 0:
        print("McNemar: b + c = 0 => no discordant pairs, test not applicable.")
    else:
        p_val = 2 * min(
            binom_dist.cdf(min(b, c), b + c, 0.5),
            1 - binom_dist.cdf(min(b, c) - 1, b + c, 0.5)
        )
        print(f"McNemar's test (Hit@6 — paired, {n_pairs} queries):")
        print(f"  b (react=✓, direct=✗) = {b}")
        print(f"  c (react=✗, direct=✓) = {c}")
        print(f"  p = {p_val:.6f}")
        if p_val < 0.05:
            print("  => Statistically significant (alpha=0.05)")
        else:
            print("  => Not significant (alpha=0.05)")

In [ ]:
# ── 8c. Wilcoxon signed-rank test on MRR and NDCG@6 ────────────────────────
from scipy.stats import wilcoxon

if not paired.empty:
    for metric, col_d, col_r in [
        ("MRR",    "reciprocal_rank_direct", "reciprocal_rank_react"),
        ("NDCG@6", "ndcg_at_6_direct",       "ndcg_at_6_react"),
    ]:
        d = paired[col_d].values
        r = paired[col_r].values
        diff = r - d
        if (diff == 0).all():
            print(f"Wilcoxon ({metric}): all differences are zero.")
            continue
        stat, p = wilcoxon(d, r, zero_method="wilcox", alternative="two-sided")
        print(f"Wilcoxon signed-rank ({metric}):")
        print(f"  Direct mean = {d.mean():.4f}  |  ReAct mean = {r.mean():.4f}")
        print(f"  W = {stat:.2f},  p = {p:.6f}")
        sig = "✓ significant" if p < 0.05 else "✗ not significant"
        print(f"  => {sig} at alpha=0.05\n")

In [ ]:
# ── 8d. Cliff's delta + Bootstrap 95% CI ────────────────────────────────────

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    n = len(x) * len(y)
    if n == 0: return np.nan
    return (np.sum(x[:, None] > y[None, :]) - np.sum(x[:, None] < y[None, :])) / n

def effect_magnitude(d):
    d = abs(d)
    if d >= 0.474: return 'large'
    if d >= 0.330: return 'medium'
    if d >= 0.147: return 'small'
    return 'negligible'

def bootstrap_ci_mean(data, n_boot=10000):
    rng = np.random.default_rng(42)
    means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    return np.quantile(means, 0.025), np.quantile(means, 0.975)

if not paired.empty:
    print("Effect sizes and Bootstrap 95% CI (ReAct vs Direct):\n")
    for metric, col_d, col_r in [
        ("MRR",    "reciprocal_rank_direct", "reciprocal_rank_react"),
        ("NDCG@6", "ndcg_at_6_direct",       "ndcg_at_6_react"),
    ]:
        d_vals = paired[col_d].values
        r_vals = paired[col_r].values
        delta = cliffs_delta(r_vals, d_vals)
        lo_d, hi_d = bootstrap_ci_mean(d_vals)
        lo_r, hi_r = bootstrap_ci_mean(r_vals)
        print(f"  {metric}:")
        print(f"    Direct:  {d_vals.mean():.4f}  95% CI [{lo_d:.4f}, {hi_d:.4f}]")
        print(f"    ReAct:   {r_vals.mean():.4f}  95% CI [{lo_r:.4f}, {hi_r:.4f}]")
        print(f"    Cliff's delta = {delta:.4f} ({effect_magnitude(delta)})\n")

## 9 · RAG Ablation Study

Isolates the contribution of each retrieval component by comparing 6 configurations:

| Variant | BM25 | SigLIP (image-vec) | Text-vec | BGE Reranker |
|---------|------|-------------------|----------|--------------|
| `bm25_only` | ✓ | ✗ | ✗ | ✗ |
| `siglip_only` | ✗ | ✓ | ✗ | ✗ |
| `text_vec_only` | ✗ | ✗ | ✓ | ✗ |
| `hybrid_no_rerank` | ✓ | ✓ | ✓ | ✗ |
| `hybrid_no_bm25` | ✗ | ✓ | ✓ | ✓ |
| `hybrid_full` ◄ | ✓ | ✓ | ✓ | ✓ |

Run: `python -m evaluation.run_rag_ablation --runs 3` from `fashion_agent/`

In [ ]:
# ── 9a. Load rag_ablation_results ───────────────────────────────────────────
q_rag = """
SELECT
    rar.retrieval_mode,
    rar.hit_at_1, rar.hit_at_3, rar.hit_at_6,
    rar.reciprocal_rank, rar.ndcg_at_6, rar.latency_ms,
    eq.difficulty, eq.category, eq.language
FROM rag_ablation_results rar
JOIN eval_queries eq ON eq.id = rar.eval_query_id
ORDER BY rar.retrieval_mode, rar.eval_query_id, rar.run_at;
"""
df_rag = pd.read_sql(q_rag, conn)
print(f"rag_ablation_results rows: {len(df_rag)}")
print(df_rag['retrieval_mode'].value_counts().to_string())
df_rag.head()

In [ ]:
# ── 9b. Aggregate metrics per retrieval variant ──────────────────────────────
MODE_ORDER = ["bm25_only", "siglip_only", "text_vec_only",
              "hybrid_no_rerank", "hybrid_no_bm25", "hybrid_full"]

if not df_rag.empty:
    rag_agg = df_rag.groupby('retrieval_mode').agg(
        n=('hit_at_6', 'count'),
        hit_at_1=('hit_at_1', 'mean'),
        hit_at_3=('hit_at_3', 'mean'),
        hit_at_6=('hit_at_6', 'mean'),
        mrr=('reciprocal_rank', 'mean'),
        ndcg_at_6=('ndcg_at_6', 'mean'),
        avg_latency_ms=('latency_ms', 'mean'),
    ).round(4)
    rag_agg = rag_agg.reindex([m for m in MODE_ORDER if m in rag_agg.index])
    print("\n━━━ RAG Ablation — Aggregate Metrics ━━━")
    display(rag_agg)
else:
    print("No rag_ablation_results yet. Run evaluation.run_rag_ablation first.")
    rag_agg = pd.DataFrame()

In [ ]:
# ── 9c. Grouped bar chart — all 6 variants ───────────────────────────────────
if not rag_agg.empty:
    metrics = ["hit_at_1", "hit_at_3", "hit_at_6", "mrr", "ndcg_at_6"]
    xlabels = ["Hit@1", "Hit@3", "Hit@6", "MRR", "NDCG@6"]
    modes = rag_agg.index.tolist()
    x = np.arange(len(metrics))
    w = 0.13
    palette = sns.color_palette("tab10", len(modes))

    fig, ax = plt.subplots(figsize=(15, 5))
    for i, mode in enumerate(modes):
        offset = (i - len(modes)/2 + 0.5) * w
        vals = [rag_agg.loc[mode, m] for m in metrics]
        hatch = "///" if mode == "hybrid_full" else None
        ax.bar(x + offset, vals, w, label=mode, color=palette[i],
               alpha=0.9, hatch=hatch)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("RAG Ablation Study — Retrieval Metrics by Component")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    if "hybrid_full" in rag_agg.index:
        ax.axhline(rag_agg.loc["hybrid_full", "ndcg_at_6"], color="gray",
                   linestyle="--", linewidth=0.8, alpha=0.6)
    plt.tight_layout()
    plt.savefig("analysis/rag_ablation_bar.png", bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# ── 9d. Component contribution: NDCG@6 delta from hybrid_full ────────────────
if not rag_agg.empty and "hybrid_full" in rag_agg.index:
    full_ndcg = rag_agg.loc["hybrid_full", "ndcg_at_6"]
    full_mrr  = rag_agg.loc["hybrid_full", "mrr"]

    deltas = []
    for mode in MODE_ORDER:
        if mode == "hybrid_full" or mode not in rag_agg.index:
            continue
        delta_ndcg = rag_agg.loc[mode, "ndcg_at_6"] - full_ndcg
        delta_mrr  = rag_agg.loc[mode, "mrr"]       - full_mrr
        deltas.append({"variant": mode, "DELTA_NDCG6": delta_ndcg, "DELTA_MRR": delta_mrr})

    df_deltas = pd.DataFrame(deltas).set_index("variant")
    print(f"\nDelta from hybrid_full (NDCG@6={full_ndcg:.4f}, MRR={full_mrr:.4f}):")
    display(df_deltas.round(4))

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in df_deltas["DELTA_NDCG6"]]
    df_deltas["DELTA_NDCG6"].plot(kind="bar", ax=ax, color=colors, edgecolor="white")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title("NDCG@6 Drop vs Full Hybrid (lower = more important component)")
    ax.set_ylabel("Delta NDCG@6  (ablated - full)")
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig("analysis/rag_component_contribution.png", bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# ── 9e. Heatmap — NDCG@6 by variant x difficulty ────────────────────────────
if not df_rag.empty:
    heat_data = df_rag.groupby(["retrieval_mode", "difficulty"])["ndcg_at_6"].mean().unstack("difficulty")
    heat_data = heat_data.reindex(
        index=[m for m in MODE_ORDER if m in heat_data.index],
        columns=["easy", "medium", "hard"]
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="YlOrRd",
              linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title("NDCG@6 Heatmap — Retrieval Variant x Query Difficulty")
    ax.set_ylabel("Retrieval Variant")
    ax.set_xlabel("Difficulty")
    plt.tight_layout()
    plt.savefig("analysis/rag_heatmap_difficulty.png", bbox_inches="tight", dpi=150)
    plt.show()

## 10 · Statistical Tests — RAG Ablation

Wilcoxon signed-rank and McNemar's tests comparing each ablation variant against  
`hybrid_full` on paired queries. Cliff's delta provides standardised effect size.

In [ ]:
# ── 10a. Build paired structure for RAG ablation ─────────────────────────────
if not df_rag.empty and "hybrid_full" in df_rag["retrieval_mode"].values:
    rag_paired_base = df_rag.groupby(["eval_query_id", "retrieval_mode"])[
        ["ndcg_at_6", "reciprocal_rank", "hit_at_6"]
    ].mean().unstack("retrieval_mode")

    full_ndcg_s = rag_paired_base["ndcg_at_6", "hybrid_full"].dropna()
    full_mrr_s  = rag_paired_base["reciprocal_rank", "hybrid_full"].dropna()
    full_hit_s  = rag_paired_base["hit_at_6", "hybrid_full"].dropna().round().astype(int)

    print(f"Paired query count (hybrid_full): {len(full_ndcg_s)}")
else:
    print("No hybrid_full data in rag_ablation_results.")
    rag_paired_base = pd.DataFrame()

In [ ]:
# ── 10b. Wilcoxon: each variant vs hybrid_full on NDCG@6 ────────────────────
from scipy.stats import wilcoxon

if not rag_paired_base.empty:
    print("Wilcoxon signed-rank vs hybrid_full (NDCG@6):\n")
    for mode in MODE_ORDER:
        if mode == "hybrid_full":
            continue
        col = ("ndcg_at_6", mode)
        if col not in rag_paired_base.columns:
            print(f"  {mode:>22}  [no data]")
            continue
        other = rag_paired_base[col].dropna()
        common = full_ndcg_s.index.intersection(other.index)
        if len(common) < 5:
            print(f"  {mode:>22}  [too few pairs: {len(common)}]")
            continue
        a = full_ndcg_s.loc[common].values
        b_vals = other.loc[common].values
        diff = a - b_vals
        if (diff == 0).all():
            print(f"  {mode:>22}  no differences")
            continue
        stat, p = wilcoxon(a, b_vals, zero_method="wilcox", alternative="two-sided")
        delta_cd = cliffs_delta(a, b_vals)
        sig = "✓" if p < 0.05 else "✗"
        print(f"  {mode:>22}  delta={a.mean()-b_vals.mean():+.4f}  W={stat:6.1f}  p={p:.4f}  {sig}  Cliffs_d={delta_cd:+.3f} ({effect_magnitude(delta_cd)})")

In [ ]:
# ── 10c. McNemar on Hit@6: each variant vs hybrid_full ──────────────────────
from scipy.stats import binom as binom_dist

if not rag_paired_base.empty:
    print("McNemar's test (Hit@6) vs hybrid_full:\n")
    for mode in MODE_ORDER:
        if mode == "hybrid_full":
            continue
        col = ("hit_at_6", mode)
        if col not in rag_paired_base.columns:
            continue
        other_hit = rag_paired_base[col].dropna().round().astype(int)
        common = full_hit_s.index.intersection(other_hit.index)
        if len(common) < 5:
            continue
        fh = full_hit_s.loc[common].values
        oh = other_hit.loc[common].values
        b_mc = ((fh == 1) & (oh == 0)).sum()
        c_mc = ((fh == 0) & (oh == 1)).sum()
        if b_mc + c_mc == 0:
            print(f"  {mode:>22}  No discordant pairs")
            continue
        p = 2 * min(
            binom_dist.cdf(min(b_mc, c_mc), b_mc + c_mc, 0.5),
            1 - binom_dist.cdf(min(b_mc, c_mc) - 1, b_mc + c_mc, 0.5)
        )
        sig = "✓" if p < 0.05 else "✗"
        print(f"  {mode:>22}  b={b_mc:2d}  c={c_mc:2d}  p={p:.4f}  {sig}")

## 11 · Combined Thesis Summary Tables

Final consolidated tables ready for LaTeX inclusion (`\\input{...csv}` via `pgfplotstable`  
or manual copy). CSVs saved to `analysis/`.

In [ ]:
# ── 11a. Pipeline comparison summary ─────────────────────────────────────────
if not eval_agg.empty:
    pipeline_summary = eval_agg.rename(columns={
        'n_queries':       'N',
        'hit_at_1':        'Hit@1',
        'hit_at_3':        'Hit@3',
        'hit_at_6':        'Hit@6',
        'mrr':             'MRR',
        'ndcg_at_6':       'NDCG@6',
        'avg_latency_ms':  'Latency(ms)',
        'avg_llm_calls':   'LLM Calls',
        'avg_total_tokens': 'Tokens',
    })
    print('\n==== PIPELINE COMPARISON (Direct vs ReAct) ====')
    display(pipeline_summary.round(4))
    pipeline_summary.to_csv('analysis/pipeline_comparison_table.csv')
    print('Saved: analysis/pipeline_comparison_table.csv')
else:
    print('No pipeline eval data yet.')

In [ ]:
# ── 11b. RAG Ablation summary ────────────────────────────────────────────────
if not rag_agg.empty:
    rag_summary = rag_agg.rename(columns={
        'n':              'N',
        'hit_at_1':       'Hit@1',
        'hit_at_3':       'Hit@3',
        'hit_at_6':       'Hit@6',
        'mrr':            'MRR',
        'ndcg_at_6':      'NDCG@6',
        'avg_latency_ms': 'Latency(ms)',
    })
    print('\n==== RAG ABLATION STUDY ====')
    display(rag_summary.round(4))
    rag_summary.to_csv('analysis/rag_ablation_table.csv')
    print('Saved: analysis/rag_ablation_table.csv')
    best_mode = rag_agg['ndcg_at_6'].idxmax()
    print(f"Best variant by NDCG@6: {best_mode} ({rag_agg.loc[best_mode, 'ndcg_at_6']:.4f})")
else:
    print('No RAG ablation data yet.')

In [ ]:
# ── 11c. List all saved figures ──────────────────────────────────────────────
import os
analysis_dir = 'analysis'
saved = [f for f in os.listdir(analysis_dir) if f.endswith('.png')]
print('Saved figures:')
for f in sorted(saved):
    print(f'  analysis/{f}')

In [ ]:
# ── Cleanup ─────────────────────────────────────────────────────────
conn.close()
print('Done.')